# Basic 3D structure search and inverse folding on multi-entity system

Please also have a look at the `antibody_engineering` example for more fine-grained control over how to add structures to a molecular system.

In [1]:
import torch
from evedesign.system import System, Protein
from evedesign.models.mpnn import LigandMPNN
from evedesign.tools.foldseek import find_structures_foldseek, filter_structures_foldseek, add_structures_foldseek
from evedesign.types import DeviceType

DEVICE: DeviceType = "cuda" if torch.cuda.is_available() else "cpu"

## Define system

For our example, we use a simple bacterial protein complex composed of two distinct chains (MoaD and MoaE).

In [2]:
# two-protein system
complex_system = System([
    Protein(
        id="moad", rep="MIKVNVLYFGAVREACDETPREEVEVQNGTDVGNLVDQLQQKYPRLRDHCQRVQMAVNQFIAPLSTVLGDGDEVAFIPQVAGG",
    ),
    Protein(
        id="moae", rep="MTQVLRAALTDQPIFLAEHEELVSHRSAGAIVGFVGMIRDRDGGRGVLRLEYSAHPSAAQVLADLVAEVAEESSGVRAVAASHRIGVLQVGEAALVAAVAADHRRAAFGTCAHLVETIKARLPVWKHQFFEDGTDEWVGSV"
    ),
])

## Find and add structures with FoldSeek

First, search for homologous structures for all the entities in the system:

In [3]:
hits = find_structures_foldseek(
    complex_system, databases=["pdb100"]
)

2026-06-19 12:28:16.174 | INFO     | evedesign.tools.foldseek:find_structures_foldseek:745 - Running foldseek for entity 0
2026-06-19 12:28:18.226 | ERROR    | evedesign.tools.foldseek:foldseek_search_sequence:165 - Sleeping for 8s. Reason: UNKNOWN
2026-06-19 12:28:35.006 | INFO     | evedesign.tools.foldseek:find_structures_foldseek:745 - Running foldseek for entity 1
2026-06-19 12:28:37.019 | ERROR    | evedesign.tools.foldseek:foldseek_search_sequence:165 - Sleeping for 6s. Reason: UNKNOWN


Then, only retain structures that cover both entities (`use_pairing = True`) and filter to top hit of these (`top_n = 1`). One could also filter the result list by identifier using the `ids` parameter:

In [4]:
hits_filt = filter_structures_foldseek(
    hits,
    use_pairing=True,   # only keep complex structures
    top_n=1  # use highest scoring one
)

Finally, add 3D structure coordinates to the molecular system (this will retrieve structures automatically and remap them to numbering consistent with the entity):

In [5]:
complex_system = add_structures_foldseek(
    complex_system, hits_filt
)

This results in two chains per entity, i.e. there are two homomultimeric copies of each chain in the PDB biological assembly:

In [6]:
complex_system[0].structures

{'6jbz': [<evedesign.structure.Structure at 0x334f95be0>,
  <evedesign.structure.Structure at 0x336cbbb30>]}

In [7]:
complex_system[1].structures

{'6jbz': [<evedesign.structure.Structure at 0x33545bd40>,
  <evedesign.structure.Structure at 0x336cbed80>]}

We can also inspect the 3D structure chain with the `res_df()` and `atom_df()` helper functions:

In [8]:
complex_system[0].structures["6jbz"][0].res_df()

,res_id,res_name,ins_code,chain_id,sse,atom_df_start_idx,res_name_oneletter
0,2,ILE,,B,C,0,I
1,3,GLN,,B,E,8,Q
2,4,VAL,,B,E,17,V
3,5,THR,,B,E,24,T
4,6,VAL,,B,E,31,V
...,...,...,...,...,...,...,...
76,79,PRO,,B,C,550,P
77,80,PHE,,B,C,557,F
78,81,ALA,,B,C,568,A
79,82,GLY,,B,C,573,G


In [9]:
complex_system[0].structures["6jbz"][0].atom_df()

,chain_id,res_id,ins_code,res_name,hetero,atom_name,element,b_factor,sym_id,charge,atom_id,occupancy,x,y,z
0,B,2,,ILE,False,N,N,35.91,0,0,1070,1.0,62.986000,17.139999,75.259003
1,B,2,,ILE,False,CA,C,33.93,0,0,1071,1.0,61.602001,17.344000,75.001999
2,B,2,,ILE,False,C,C,39.18,0,0,1072,1.0,61.245998,16.663000,73.717003
3,B,2,,ILE,False,O,O,37.62,0,0,1073,1.0,62.066002,16.420000,72.884003
4,B,2,,ILE,False,CB,C,34.92,0,0,1074,1.0,61.259998,18.813999,74.905998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
576,B,82,,GLY,False,O,O,37.32,0,0,1657,1.0,47.026001,26.513000,42.292999
577,B,83,,GLY,False,N,N,40.23,0,0,1658,1.0,44.938000,26.007999,42.768002
578,B,83,,GLY,False,CA,C,37.79,0,0,1659,1.0,44.472000,26.246000,41.416000
579,B,83,,GLY,False,C,C,33.53,0,0,1660,1.0,44.803001,25.044001,40.544998


## Build model and generate new designs

With the added 3D structure information, we can now build an inverse folding model on the molecular system:

In [ ]:
# generate sequences on complex with ProteinMPNN
mpnn = LigandMPNN(
    model_name="solublempnn_v_48_002",
    batch_size=16,
    device=DEVICE,
).build(complex_system)

Then, generate new sequences on the 3D structure backbone for both entities.

In [11]:
# generate from the model
designs = mpnn.generate(
    num_designs=16, temperature=0.5, entities=[0, 1]
)

In [12]:
designs

[SystemInstance([EntityInstance(rep=MMRVRVRFFAAARAAAGEAASEDVELAEGATVRDLIDALRNRNPELARVL..., models=None), EntityInstance(rep=ATRVEAAELTDAPIHMAEYIELVRDEHAGAIVASEHRVRSELDGKKIKSV..., models=None)] id=None score=0.9175341129302979),
 SystemInstance([EntityInstance(rep=MINVTVRFYAAAQDAAGETASQIVRLEEGSTVKDLVDKLARQNPELAKVL..., models=None), EntityInstance(rep=ATRVEAATLTDEPISAQEHIDLVRDANAGAIVSFESIVPATKDGKKIKSV..., models=None)] id=None score=0.8524802923202515),
 SystemInstance([EntityInstance(rep=MMTVEVRFYAAAEEAAGETASRLISLEEGATVRELVAELARENPRLARVL..., models=None), EntityInstance(rep=QTRVEAAELTDEPIFMEEHVELVRDDRAGAIVATERLVRSELGGKRVASV..., models=None)] id=None score=0.9061375856399536),
 SystemInstance([EntityInstance(rep=MMKVTVRFYAAAQDAAGETASKEVTLEDGATVADLIEKLSKENADLAKVL..., models=None), EntityInstance(rep=ATEVEAALLTREPISQAEFDDLVRHPEAGAIVSYESRVPATLDGREVERV..., models=None)] id=None score=0.9417998790740967),
 SystemInstance([EntityInstance(rep=MLSVTVRYYAAAADAAGEKESEVVSLKEGATVKDLIDLLSQRSEDLAEVL..